# 자동화형 Agent: GitHub 이슈 일일 보고

PydanticAI를 활용하여 GitHub 이슈를 수집하고 Slack으로 요약 보고를 발송하는 **자동화형 Agent**입니다.

**자동화형 Agent의 특징:**
- 정해진 도구를 **순서대로 호출**하여 반복 업무를 자동화
- 이슈 수집(fetch) → 분석 → 발송(send)의 **고정된 워크플로우**
- `output_type`으로 구조화된 보고서를 반환받아 후속 처리 가능

## 환경 설정

`.env` 파일에서 API 키를 로드하고, PydanticAI Agent 구성에 필요한 라이브러리를 임포트합니다.

In [1]:
# .env 파일에서 OPENAI_API_KEY 등 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

from datetime import datetime
from pydantic import BaseModel      # 구조화된 출력 스키마 정의용
from pydantic_ai import Agent       # LLM 기반 Agent 프레임워크

## Structured Output

Agent가 반환할 일일 보고서의 구조를 Pydantic 모델로 정의합니다.
`output_type`에 이 모델을 지정하면 LLM이 자유 텍스트 대신 **정해진 스키마에 맞는 JSON**을 반환합니다.

In [2]:
# Agent가 반환할 일일 보고서 스키마
class DailyReport(BaseModel):
    summary: str          # 전체 요약
    issue_count: int      # 이슈 건수
    highlights: list[str] # 주요 하이라이트 목록

## Agent 정의

자동화형 Agent는 `instructions`에 도구 사용 순서를 명시합니다.
LLM이 이 지시사항을 따라 fetch → 분석 → send 순서로 도구를 호출합니다.

In [3]:
# 자동화형 Agent: 도구 호출 순서를 instructions로 명시
agent = Agent(
    "openai:gpt-5.4",
    output_type=DailyReport,  # 구조화된 보고서 형식으로 반환 강제
    instructions=(
        "개발팀 일일 보고를 작성하는 어시스턴트. "
        "fetch_github_issues로 이슈를 수집하고, 분석한 뒤, "
        "send_slack_message로 Slack에 요약을 발송하라. "
        "최종 출력은 DailyReport 형식으로 반환."
    ),
)

## Tools (실제 API + Fallback)

GitHub REST API와 Slack Bot API를 호출하는 **실제 도구**를 정의합니다.
API 키가 없거나 호출에 실패하면 자동으로 **Fallback 데이터**를 반환하므로,
환경 설정 없이도 실습이 가능합니다.

- `fetch_github_issues`: GitHub REST API로 이슈 수집 (토큰 없으면 공개 저장소만 가능)
- `send_slack_message`: Slack Bot API로 메시지 발송 (`SLACK_BOT_TOKEN` 미설정 시 콘솔 출력)

`@agent.tool_plain`은 Agent 컨텍스트 없이 독립적으로 동작하는 도구를 정의하는 데코레이터입니다.

In [4]:
# GitHub 이슈 수집 도구: GitHub REST API 호출 (실패 시 Fallback)
@agent.tool_plain
def fetch_github_issues(repo: str) -> str:
    """GitHub 저장소의 최근 이슈 목록을 가져온다."""
    import httpx
    import os
    try:
        headers = {"Accept": "application/vnd.github.v3+json"}
        token = os.getenv("GITHUB_TOKEN")
        if token:
            headers["Authorization"] = f"Bearer {token}"
        resp = httpx.get(
            f"https://api.github.com/repos/{repo}/issues",
            headers=headers,
            params={"state": "open", "per_page": 5, "sort": "updated"},
            timeout=10,
        )
        resp.raise_for_status()
        issues = resp.json()
        lines = []
        for issue in issues:
            labels = ",".join(l["name"] for l in issue.get("labels", []))
            lines.append(
                f"#{issue['number']} [{labels}] {issue['title']} ({issue['user']['login']})"
            )
        print(f"  [Tool] fetch_github_issues('{repo}') → {len(lines)}건 (GitHub API)")
        return "\n".join(lines) if lines else "열린 이슈가 없습니다."
    except Exception as e:
        print(f"  [Tool] fetch_github_issues('{repo}') → Fallback 사용 ({e})")
        fallback = [
            "#142 [bug] 로그인 API 간헐적 500 에러 (alice)",
            "#138 [feature] 대시보드 활동 위젯 추가 (bob)",
            "#135 [refactor] CI 빌드 최적화 (charlie)",
        ]
        return "\n".join(fallback)


# Slack 메시지 발송 도구: Slack Bot API 호출 (토큰 미설정 시 콘솔 출력)
@agent.tool_plain
def send_slack_message(channel: str, message: str) -> str:
    """Slack 채널에 메시지를 발송한다."""
    import httpx
    import os
    slack_token = os.getenv("SLACK_BOT_TOKEN")
    # 환경변수에서 채널 가져오기 (SLACK_CHANNEL_ID 또는 SLACK_CHANNEL)
    slack_channel = (
        os.getenv("SLACK_CHANNEL_ID")
        or os.getenv("SLACK_CHANNEL")
        or channel
    ).lstrip("#")
    print(f"  [Tool] send_slack_message('#{slack_channel}')")
    print(f"  ---- 메시지 내용 ----\n{message}\n  --------------------")
    if slack_token:
        try:
            resp = httpx.post(
                "https://slack.com/api/chat.postMessage",
                headers={"Authorization": f"Bearer {slack_token}"},
                json={"channel": slack_channel, "text": message},
                timeout=10,
            )
            data = resp.json()
            if data.get("ok"):
                print(f"  [Slack] 실제 발송 완료")
                return f"#{slack_channel} Slack 발송 완료"
            else:
                print(f"  [Slack] API 오류: {data.get('error')}")
        except Exception as e:
            print(f"  [Slack] 발송 실패 ({e})")
    else:
        print(f"  [Slack] SLACK_BOT_TOKEN 미설정 → 콘솔 출력으로 대체")
    return f"#{slack_channel} 메시지 출력 완료 (콘솔)"

## 실행

Agent에게 작업을 지시하면 `fetch_github_issues` → `send_slack_message` 순서로 도구를 호출합니다.
결과는 `DailyReport` 형식의 구조화된 데이터로 반환됩니다.

> **참고:** Jupyter 노트북은 이미 이벤트 루프가 실행 중이므로 `run_sync()` 대신 `await agent.run()`을 사용합니다.

In [5]:
f"{datetime.now():%Y-%m-%d} - Agent 실행 시작"

'2026-03-23 - Agent 실행 시작'

In [6]:
# Agent 실행 — Jupyter에서는 await 사용 (run_sync는 이벤트 루프 충돌 발생)
result = await agent.run(
    f"오늘({datetime.now():%Y-%m-%d}) langchain-ai/langchain 저장소의 "
    "이슈를 수집하고 #dev-daily 채널에 요약을 보내주세요."
)
report = result.output  # DailyReport 타입의 구조화된 결과

# 구조화된 보고서 내용 출력
print(f"[결과] 요약: {report.summary}")
print(f"[결과] 이슈 수: {report.issue_count}")
print(f"[결과] 하이라이트:")
for h in report.highlights:
    print(f"  - {h}")

  [Tool] fetch_github_issues('langchain-ai/langchain') → 5건 (GitHub API)
  [Tool] send_slack_message('#dev-daily')
  ---- 메시지 내용 ----
[2026-03-23] langchain-ai/langchain 일일 이슈 요약
- 총 5건 확인
- 주요 내용:
  1. #35378 fix(core): `get_from_dict_or_env`에서 문자열 강제 변환 제거
  2. #35295 feat: tracing metadata에 패키지 버전 추적 추가
  3. #34469 제안: “No Side Effect Without Provenance” invariant
  4. #35615 fix(core): streaming usage_metadata 병합 시 토큰 수 과대계상 방지
  5. #35630 fix(core): `trim_messages`의 per-message token_counter 분류 오류 수정
- 요약: core 영역의 버그 수정 3건, 추적 메타데이터 기능 추가 1건, 아키텍처/안전성 관련 제안 1건이 있었습니다. 오늘은 토큰 집계 정확도와 메시지 처리 안정성 개선이 두드러졌습니다.
  --------------------
  [Slack] SLACK_BOT_TOKEN 미설정 → 콘솔 출력으로 대체
[결과] 요약: 2026-03-23 기준 langchain-ai/langchain 저장소의 최근 이슈 5건을 수집해 분석했고, #dev-daily 채널로 요약을 발송했습니다. 핵심은 core 관련 버그 수정 3건, tracing metadata 기능 추가 1건, 아키텍처/안전성 제안 1건입니다.
[결과] 이슈 수: 5
[결과] 하이라이트:
  - #35378 `get_from_dict_or_env` 문자열 강제 변환 제거
  - #35295 tracing metadata에 패키지 버전 추적 추가
  - #35615 streaming usage_metadat